# Lab 1 — The brochure generator

**~45 minutes.** This is Ed Donner's Week 1 project from *LLM Engineering*, rebuilt on the
free stack. You scrape a company website, let the model decide which pages matter, fetch
those, and generate a sales brochure.

It is a small program, but it is the shape of most real LLM features: **fetch → let the
model choose → fetch again → generate**. Two model calls, chained, with your code in
control between them.

In [ ]:
import json
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

from shared import ask, chat, stream, extract_json

HEADERS = {"User-Agent": "Mozilla/5.0 (workshop lab; polite scraper)"}


def fetch(url: str) -> dict:
    """Return {url, title, text, links} for a page. Deliberately crude — 20 lines is enough."""
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.content, "html.parser")
    title = soup.title.string.strip() if soup.title and soup.title.string else "(no title)"
    for tag in soup(["script", "style", "img", "input", "svg", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    links = [a.get("href") for a in soup.find_all("a") if a.get("href")]
    return {"url": url, "title": title, "text": text[:6000], "links": links}


page = fetch("https://anthropic.com")
print(page["title"], "|", len(page["text"]), "chars |", len(page["links"]), "links")
print(page["text"][:400])

## 1. First call — let the model pick the links

A landing page links to fifty things; three of them matter for a brochure. Rather than
writing a heuristic, describe the job and ask for **JSON you can parse**.

Two rules that carry over to every extraction prompt you will ever write:

- show the exact output shape you want, as an example,
- tell it what to leave out (privacy policies, careers-page boilerplate, mailto: links).

In [ ]:
LINK_SYSTEM = """You are given a list of links found on a company website.
Return only the links that belong in a customer-facing sales brochure: about/company pages,
product or platform pages, customer or case-study pages, pricing, and the blog index.

Exclude: privacy policies, terms, cookie notices, careers/jobs, login pages, social media,
mailto: links, and in-page anchors.

Respond with JSON only, no prose, in exactly this shape:
{"links": [{"type": "about page", "url": "https://full.url/about"}]}
Use absolute URLs. Return at most 5 links."""


def choose_links(page: dict) -> list[dict]:
    user = (
        f"Website: {page['url']}\n"
        f"Links found on the page (some are relative — make them absolute):\n"
        + "\n".join(page["links"][:100])
    )
    raw = ask(user, system=LINK_SYSTEM, temperature=0)
    return extract_json(raw)["links"]


chosen = choose_links(page)
chosen

**If that failed to parse:** good, that is the lesson. Free models wrap JSON in prose or a
code fence about a third of the time. `extract_json()` in `shared.py` handles the common
cases; production code uses the provider's structured-output mode instead. Lab 2 makes this
robust properly — for now, re-run, or lower the temperature.

## 2. Fetch the chosen pages

Your code, not the model, does the fetching. Note the failure handling: an agent that
crashes on one 404 is useless, so collect what works and move on.

In [ ]:
def gather(page: dict, chosen: list[dict], limit: int = 4) -> str:
    parts = [f"### landing page — {page['url']}\n{page['text']}"]
    for link in chosen[:limit]:
        try:
            sub = fetch(link["url"])
        except Exception as exc:
            print(f"  skipped {link['url']}: {type(exc).__name__}")
            continue
        parts.append(f"### {link['type']} — {sub['url']}\n{sub['text'][:3000]}")
        print(f"  fetched {link['url']} ({len(sub['text'])} chars)")
    return "\n\n".join(parts)


corpus = gather(page, chosen)
print(len(corpus), "chars of context")

## 3. Second call — write the brochure

Now the generation call. Give it a role, an audience, a structure, and a length limit.
Stream it, because a brochure takes a while and a blank screen feels broken.

In [ ]:
BROCHURE_SYSTEM = """You are a B2B copywriter producing a short company brochure for
prospective customers and potential hires.

Write markdown with: a title, a one-line positioning statement, then the sections
'What they do', 'Who it is for', and 'Why it stands out'.

Ground every claim in the supplied page text. If something is not in the text, leave it
out — do not infer funding, headcount, or customers. Around 300 words."""


text = ""
for piece in stream(f"Here is the scraped site content:\n\n{corpus}",
                    system=BROCHURE_SYSTEM, temperature=0.4):
    text += piece
    print(piece, end="", flush=True)

In [ ]:
display(Markdown(text))

## 4. Compare against the one-shot version

Run the same job as a single call — dump the landing page in and ask for a brochure, with no
link-picking step. Read both. The chained version is better because *your code* did the
retrieval; the model only did the judging and the writing.

That contrast is the whole argument for Lab 4 (RAG) and Lab 5 (agents).

In [ ]:
one_shot = ask(f"Here is a company landing page:\n\n{page['text']}",
               system=BROCHURE_SYSTEM, temperature=0.4)
display(Markdown(one_shot))

## Stretch goals

1. **Tone switch.** Add a `tone` parameter — "formal", "snarky", "recruiter" — and re-run
   without re-fetching. Cheap to try, and it shows how much of the product is the prompt.
2. **A different site.** Point it at your own company. Read the brochure critically: which
   claims are grounded, and which did it invent? Write down the invented ones — those are
   your first eval cases.
3. **Cost.** Count the tokens you sent across both calls (`tiktoken`, as in Lab 0) and price
   it at a paid model's rate. Then price the one-shot version. Which would you ship?
4. **Translate.** Add a third call that renders the brochure in another language, and check
   whether the grounding survives translation.